### Die goldene Mitte – Lösung Schritt für Schritt

*42. Bundeswettbewerb Informatik (2023), 1. Runde, Aufgabe 2*

Dirk hat zum Geburtstag ein kniffliges Rätsel geschenkt bekommen. Das Rätsel besteht aus einer würfelförmigen Kiste
mit einer inneren Kantenlänge von 5 cm und einer Menge von Würfeln und weiteren Quadern. Darunter ist ein goldener
Würfel mit Kantenlänge 1 cm.

Die Würfel und Quader sollen im Inneren der Kiste so platziert werden, dass sie diese komplett füllen.
Dabei muss der goldene Würfel genau in der Mitte platziert werden.

Zur Verfügung stehen neben dem goldenen Würfel weitere vier Würfel der Kantenlänge 1 cm, sechs Quader mit den Maßen
1 cm × 2 cm × 4 cm und noch einmal sechs Quader mit den Kantenlängen 2 cm × 2 cm × 3 cm.

#### Aufgabe 2

Hilf Dirk und schreibe ein Programm, das die Kiste mit den gegebenen Würfeln und Quadern nach den obigen Regeln füllt –
oder meldet, dass es keine Lösung gibt.

Dein Programm soll auch für andere Kisten und Mengen von Würfeln und Quadern funktionieren. Dabei sind die Kantenlängen
immer ganzzahlig, und auch ein goldener Würfel ist immer dabei. Er soll, wie oben, immer genau in der Mitte platziert werden.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest,
und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/raetsel1.txt` bis `raetsel5.txt`)

- 1. Zeile: innere Kantenlängen der Kiste `x y z`
- 2. Zeile: Anzahl `n` der Würfel und Quader
- folgende `n` Zeilen: Kantenlängen `a b c` je eines Würfels oder Quaders

Der goldene Würfel steht **nicht** in den Dateien, er ist immer zusätzlich dabei. `raetsel5.txt` ist das Beispiel aus der
Aufgabenstellung. Als Ausgabe bietet es sich an, die Kiste Ebene für Ebene darzustellen, mit der Nummer des Quaders in jedem
Feld und `G` für den goldenen Würfel.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Volumen vergleichen:** Passen die Bausteine überhaupt? Das Volumen der Kiste muss gleich der Summe der Baustein-Volumina
  **plus 1** (der goldene Würfel) sein. Sonst kann man sofort „keine Lösung“ melden.
- **Rückverfolgung (Backtracking):** Einen Baustein hinlegen, mit dem Rest weitermachen, und wenn es nicht weitergeht, den
  Baustein wieder wegnehmen und etwas anderes probieren. Das ist rekursiv: Die Funktion ruft sich für den Rest der Kiste selbst auf.
- **Immer die erste freie Stelle füllen:** Jede Stelle der Kiste muss irgendwann von einem Baustein belegt werden. Wenn wir die Zellen
  in fester Reihenfolge durchgehen, ist die erste freie Zelle immer die **Ecke** des nächsten Bausteins (davor ist alles belegt).
  Dadurch müssen wir nur die Bausteine und ihre Drehungen ausprobieren, nicht jede mögliche Verschiebung.
- **Doppelte Arbeit vermeiden:** Sind zwei Bausteine gleich, bringt es nichts, beide an derselben Stelle zu probieren. Und ein
  Baustein mit gleichen Seiten (z. B. `2 2 3`) hat weniger als sechs verschiedene Drehungen.

**Nützliche Funktionen und Bibliotheken**
- `itertools.permutations`: alle Reihenfolgen von `(a, b, c)`, das sind die Drehungen eines Quaders
- `set(...)`: wirft doppelte Einträge weg (z. B. bei Würfeln und Quadern mit zwei gleichen Seiten)
- dreifach verschachtelte Listen `gitter[z][y][x]` für die Kiste; Rekursion

## Die Idee

1. **Einlesen** und gleich prüfen: Hat die Kiste eine Mitte, und stimmt das Volumen?
2. Die Kiste als **dreidimensionales Gitter** speichern: `0` = frei, `-1` = goldener Würfel, `k` = Baustein Nummer `k`.
3. Hilfsfunktionen: **passt** ein Quader an eine Stelle? **Setzen** und wieder **entfernen**.
4. Die **Suche**: erste freie Zelle suchen, alle noch nicht benutzten Bausteine in allen Drehungen probieren, rekursiv weitermachen.
5. Die Kiste **Ebene für Ebene ausgeben** und die Lösung **automatisch prüfen**.
6. Alle Beispiele durchrechnen.

## Schritt 1: Einlesen und Voraussetzungen prüfen

Die Datei liefert die Kiste `(x, y, z)` und die Liste der Bausteine, jeder als Tupel `(a, b, c)`. Der goldene Würfel steht
nicht darin; wir behandeln ihn getrennt.

Der goldene Würfel soll **genau in der Mitte** stehen. Bei einer Kantenlänge `x` ist das die Zelle `x // 2` (gezählt ab 0).
Das ist aber nur dann wirklich die Mitte, wenn `x` **ungerade** ist: Bei `x = 4` gibt es keine mittlere Zelle. Dann
gibt es auch keine Lösung.

In [ ]:
def lies_raetsel(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    kiste = tuple(int(wort) for wort in zeilen[0].split())
    anzahl = int(zeilen[1])
    bausteine = []
    for i in range(anzahl):
        bausteine.append(tuple(int(wort) for wort in zeilen[2 + i].split()))
    return kiste, bausteine


def vorpruefung(kiste, bausteine):
    """Gibt None zurück, wenn alles in Ordnung ist, sonst den Grund, warum es keine Lösung geben kann."""
    for kante in kiste:
        if kante % 2 == 0:
            return 'Die Kante ' + str(kante) + ' ist gerade, die Kiste hat keine Mitte.'
    volumen_bausteine = 1                      # der goldene Würfel
    for a, b, c in bausteine:
        volumen_bausteine = volumen_bausteine + a * b * c
    volumen_kiste = kiste[0] * kiste[1] * kiste[2]
    if volumen_kiste != volumen_bausteine:
        return 'Volumen der Kiste ' + str(volumen_kiste) + ', Volumen der Bausteine mit Gold ' + str(volumen_bausteine) + '.'
    return None


kiste, bausteine = lies_raetsel('beispieldaten/raetsel5.txt')
print(kiste)
print(bausteine)
print(vorpruefung(kiste, bausteine))

(5, 5, 5)
[(1, 1, 1), (2, 2, 3), (2, 2, 3), (2, 2, 3), (2, 2, 3), (2, 2, 3), (2, 2, 3), (1, 2, 4), (1, 2, 4), (1, 2, 4), (1, 2, 4), (1, 2, 4), (1, 2, 4), (1, 1, 1), (1, 1, 1), (1, 1, 1)]
None


Die Kiste hat $5 \cdot 5 \cdot 5 = 125$ cm³. Die Bausteine: $4 \cdot 1 + 6 \cdot 8 + 6 \cdot 12 = 124$, dazu der goldene Würfel: 125. Das passt.

## Schritt 2: Das Gitter und die Drehungen

`neues_gitter` baut die Kiste als Liste von Ebenen (`z`), jede Ebene eine Liste von Zeilen (`y`), jede Zeile eine Liste von Zellen
(`x`). Zugriff also mit `gitter[z][y][x]`. In die Mitte kommt `-1` für das Gold.

Ein Quader `(a, b, c)` kann auf sechs Arten liegen: Welche Seite zeigt in x-, welche in y-, welche in z-Richtung? Das sind die
Reihenfolgen von `a, b, c`. `permutations` liefert sie alle, `set` entfernt doppelte, wenn Seiten gleich lang sind.

In [ ]:
from itertools import permutations


def neues_gitter(kiste):
    laenge, breite, hoehe = kiste
    gitter = []
    for z in range(hoehe):
        ebene = []
        for y in range(breite):
            ebene.append([0] * laenge)
        gitter.append(ebene)
    gitter[hoehe // 2][breite // 2][laenge // 2] = -1       # der goldene Würfel
    return gitter


def drehungen(baustein):
    return sorted(set(permutations(baustein)))


print(drehungen((1, 2, 4)))
print(drehungen((2, 2, 3)))
print(drehungen((1, 1, 1)))

[(1, 2, 4), (1, 4, 2), (2, 1, 4), (2, 4, 1), (4, 1, 2), (4, 2, 1)]
[(2, 2, 3), (2, 3, 2), (3, 2, 2)]
[(1, 1, 1)]


Bei `(1, 2, 4)` sind es sechs Drehungen, bei `(2, 2, 3)` nur drei, beim Würfel genau eine.

## Schritt 3: Passt ein Quader an diese Stelle?

Ein Quader der Maße `a × b × c` mit der Ecke bei `(x, y, z)` belegt die Zellen von `x` bis `x + a - 1` und so weiter. Er passt,
wenn

1. er hinten nicht aus der Kiste ragt (`x + a <= laenge` usw.) und
2. **alle** seine Zellen noch frei (`0`) sind.

`setze` schreibt eine Nummer in alle Zellen des Quaders. Zum Entfernen schreibt man einfach wieder `0` hinein.

In [ ]:
def passt(gitter, x, y, z, a, b, c):
    hoehe = len(gitter)
    breite = len(gitter[0])
    laenge = len(gitter[0][0])
    if x + a > laenge or y + b > breite or z + c > hoehe:
        return False
    for zz in range(z, z + c):
        for yy in range(y, y + b):
            for xx in range(x, x + a):
                if gitter[zz][yy][xx] != 0:
                    return False
    return True


def setze(gitter, x, y, z, a, b, c, wert):
    for zz in range(z, z + c):
        for yy in range(y, y + b):
            for xx in range(x, x + a):
                gitter[zz][yy][xx] = wert


gitter = neues_gitter((3, 3, 3))
print(passt(gitter, 0, 0, 0, 1, 1, 3))     # ganz links unten, passt
setze(gitter, 0, 0, 0, 1, 1, 3, 1)         # Baustein Nummer 1 hinlegen
print(passt(gitter, 0, 0, 0, 1, 1, 3))     # die Zellen sind jetzt belegt
print(passt(gitter, 1, 1, 0, 1, 1, 3))     # würde durch das Gold in der Mitte gehen
print(passt(gitter, 2, 2, 1, 1, 1, 3))     # ragt oben heraus

True
False
False
False


## Schritt 4: Die Suche

Die Suche ist eine Funktion, die sich selbst aufruft:

1. Suche die **erste freie Zelle** (Ebene für Ebene, Zeile für Zeile). Gibt es keine, ist die Kiste voll: **Lösung gefunden**.
2. Probiere jeden **noch nicht benutzten Baustein** in jeder **Drehung** mit der Ecke an dieser Zelle.
   Passt er, lege ihn hin, markiere ihn als benutzt und rufe die Suche für den Rest auf.
   Findet der Aufruf eine Lösung, gib `True` weiter. Sonst nimm den Baustein **wieder weg**.
3. Hat nichts gepasst, gib `False` zurück. Der Aufrufer davor probiert dann etwas anderes.

Gleiche Bausteine (z. B. die sechs `1 2 4`) probieren wir pro Stelle nur einmal: Wir merken uns in `probiert`, welche
Maße schon dran waren. Das ändert nichts am Ergebnis, spart aber viel Arbeit. Mit dem Schalter `gleiche_auslassen` kann man das
später ausschalten und vergleichen.

Im Wörterbuch `zaehler` zählen wir die Aufrufe der Suche (keine Laufzeit, die wäre nicht reproduzierbar).

In [ ]:
zaehler = {'aufrufe': 0}


def erste_freie_zelle(gitter):
    for z in range(len(gitter)):
        for y in range(len(gitter[z])):
            for x in range(len(gitter[z][y])):
                if gitter[z][y][x] == 0:
                    return x, y, z
    return None


def suche(gitter, bausteine, benutzt, gleiche_auslassen=True):
    zaehler['aufrufe'] = zaehler['aufrufe'] + 1
    stelle = erste_freie_zelle(gitter)
    if stelle is None:
        return True                           # alles gefüllt
    x, y, z = stelle
    probiert = []
    for nummer in range(len(bausteine)):
        if benutzt[nummer]:
            continue
        if gleiche_auslassen and bausteine[nummer] in probiert:
            continue
        probiert.append(bausteine[nummer])
        for a, b, c in drehungen(bausteine[nummer]):
            if passt(gitter, x, y, z, a, b, c):
                setze(gitter, x, y, z, a, b, c, nummer + 1)
                benutzt[nummer] = True
                if suche(gitter, bausteine, benutzt, gleiche_auslassen):
                    return True
                setze(gitter, x, y, z, a, b, c, 0)
                benutzt[nummer] = False
    return False


def loese(kiste, bausteine, gleiche_auslassen=True):
    """Gibt das gefüllte Gitter zurück oder None, wenn es keine Lösung gibt."""
    if vorpruefung(kiste, bausteine) is not None:
        return None
    gitter = neues_gitter(kiste)
    benutzt = [False] * len(bausteine)
    zaehler['aufrufe'] = 0
    if suche(gitter, bausteine, benutzt, gleiche_auslassen):
        return gitter
    return None

Ein Detail: `x, y, z` der ersten freien Zelle ist wirklich die **kleinste** Ecke des nächsten Bausteins, denn alle Zellen davor
sind belegt. Wir müssen deshalb keine Verschiebungen probieren, nur Bausteine und Drehungen.

Zwei Bausteine, die nur in anderer Reihenfolge geschrieben sind (z. B. `(1, 2, 4)` und `(2, 1, 4)`), zählt `probiert` als verschieden.
Das kostet nur etwas Zeit und ändert das Ergebnis nicht.

## Schritt 5: Ausgabe und automatische Prüfung

Wie in der README: Die Ebenen stehen nebeneinander, jede Zelle zeigt die Nummer des Bausteins (in der Reihenfolge der Datei, ab 1),
`G` ist das Gold.

Damit wir der Lösung trauen, prüft `pruefe_loesung` alles nach, unabhängig von der Suche: Keine Zelle ist frei, genau eine
Zelle ist golden und sie liegt in der Mitte, und für **jeden** Baustein bilden seine Zellen einen Quader mit den richtigen Maßen.

In [ ]:
def zeige(gitter):
    hoehe = len(gitter)
    kopf = ''
    for z in range(hoehe):
        kopf = kopf + ('Ebene ' + str(z)).ljust(3 * len(gitter[0][0]) + 3)
    print(kopf)
    for y in range(len(gitter[0])):
        zeile = ''
        for z in range(hoehe):
            for x in range(len(gitter[z][y])):
                wert = gitter[z][y][x]
                if wert == -1:
                    zeile = zeile + ' G '
                else:
                    zeile = zeile + str(wert).rjust(2) + ' '
            zeile = zeile + '   '
        print(zeile)


def pruefe_loesung(kiste, bausteine, gitter):
    laenge, breite, hoehe = kiste
    zellen = {}                                   # Nummer -> Liste der Zellen
    for z in range(hoehe):
        for y in range(breite):
            for x in range(laenge):
                wert = gitter[z][y][x]
                if wert == 0:
                    return False                  # eine freie Zelle
                if wert not in zellen:
                    zellen[wert] = []
                zellen[wert].append((x, y, z))
    if zellen[-1] != [(laenge // 2, breite // 2, hoehe // 2)]:
        return False                              # Gold nicht allein in der Mitte
    if len(zellen) != len(bausteine) + 1:
        return False                              # nicht jeder Baustein genau einmal
    for nummer in range(1, len(bausteine) + 1):
        liste = zellen[nummer]
        xs = [p[0] for p in liste]
        ys = [p[1] for p in liste]
        zs = [p[2] for p in liste]
        a = max(xs) - min(xs) + 1
        b = max(ys) - min(ys) + 1
        c = max(zs) - min(zs) + 1
        if len(liste) != a * b * c:
            return False                          # kein voller Quader
        if sorted((a, b, c)) != sorted(bausteine[nummer - 1]):
            return False                          # falsche Maße
    return True


kiste3, bausteine3 = lies_raetsel('beispieldaten/raetsel1.txt')
loesung3 = loese(kiste3, bausteine3)
zeige(loesung3)
print(pruefe_loesung(kiste3, bausteine3, loesung3))
print('Aufrufe:', zaehler['aufrufe'])

Ebene 0     Ebene 1     Ebene 2     
 1  3  2     1  3  4     1  8  8    
 5  5  2     6  G  4     6  8  8    
 5  5  2     6  7  7     6  7  7    
True
Aufrufe: 143


## Schritt 6: Alle Beispiele

Für jedes Beispiel: Kiste einlesen, lösen, ausgeben, prüfen. Bei `raetsel3` gibt es keine Lösung: Alle Möglichkeiten werden
durchprobiert (das sind die Aufrufe) und keine passt.

In [ ]:
ergebnisse = {}
for nummer in range(1, 6):
    k, b = lies_raetsel('beispieldaten/raetsel' + str(nummer) + '.txt')
    g = loese(k, b)
    ergebnisse[nummer] = (k, b, g, zaehler['aufrufe'])
    print('=== raetsel' + str(nummer) + ': Kiste', k, ',', len(b), 'Bausteine')
    if g is None:
        print('Keine Lösung.', '(Aufrufe:', zaehler['aufrufe'], ')')
    else:
        zeige(g)
        print('Prüfung:', pruefe_loesung(k, b, g), '(Aufrufe:', zaehler['aufrufe'], ')')
    print()

=== raetsel1: Kiste (3, 3, 3) , 8 Bausteine
Ebene 0     Ebene 1     Ebene 2     
 1  3  2     1  3  4     1  8  8    
 5  5  2     6  G  4     6  8  8    
 5  5  2     6  7  7     6  7  7    
Prüfung: True (Aufrufe: 143 )

=== raetsel2: Kiste (3, 3, 3) , 6 Bausteine
Ebene 0     Ebene 1     Ebene 2     
 1  5  2     1  5  2     1  5  2    
 1  3  2     1  G  2     1  4  2    
 1  6  2     1  6  2     1  6  2    
Prüfung: True (Aufrufe: 19 )

=== raetsel3: Kiste (3, 3, 3) , 6 Bausteine
Keine Lösung. (Aufrufe: 393 )

=== raetsel4: Kiste (5, 5, 5) , 20 Bausteine
Ebene 0           Ebene 1           Ebene 2           Ebene 3           Ebene 4           
 1  2  2  3  3     1  2  2  3  3     1 11 16 17 17     1 12 16  5  5    13 14 16  5  5    
 1  2  2  3  3     1  2  2  3  3     1 11 16 17 17     1 12 16  5  5    13 14 16  5  5    
 4  4 15 15 15     4  4 15 15 15    18 11  G 17 17    18 12 19 19 19    13 14 19 19 19    
 4  4  6  7  8     4  4  6  7  8    18 11  6  7  8    18 12  6  7  8   

**Ergebnis:** Die Beispiele 1, 2, 4 und 5 sind lösbar, Beispiel 3 nicht. Das stimmt mit den Lösungshinweisen überein. Die
Lösungen sind nicht eindeutig; unsere sehen anders aus als die Schnittbilder der Hinweise, sind aber nach `pruefe_loesung` gültig.

**Warum hat `raetsel3` keine Lösung?** Das Volumen stimmt ($1 + 1 + 6 + 8 + 6 + 4 + 1 = 27$), aber die Bausteine passen nicht zusammen,
das fand die Suche durch vollständiges Probieren heraus.

## Eigene Rätsel und Sonderfälle

Die Aufgabe will auch andere Kisten. Wir bauen die Eingabe direkt in Python:

In [ ]:
# Kiste mit gerader Kante: keine Mitte
print(vorpruefung((4, 3, 3), [(1, 1, 1)]))
print(loese((4, 3, 3), [(1, 1, 1)]))

# Volumen stimmt nicht
print(vorpruefung((3, 3, 3), [(1, 1, 1), (3, 3, 3)]))

# nicht würfelförmige Kiste 3 x 3 x 1: Gold in der Mitte, außen herum ein Ring aus zwei langen Quadern und zwei Würfeln
ring = [(3, 1, 1), (3, 1, 1), (1, 1, 1), (1, 1, 1)]
print(vorpruefung((3, 3, 1), ring))
ring_loesung = loese((3, 3, 1), ring)
zeige(ring_loesung)
print(pruefe_loesung((3, 3, 1), ring, ring_loesung))

Die Kante 4 ist gerade, die Kiste hat keine Mitte.
None
Volumen der Kiste 27, Volumen der Bausteine mit Gold 29.
None
Ebene 0     
 1  3  2    
 1  G  2    
 1  4  2    
True


## Für Neugierige: Wie viel spart das Auslassen gleicher Bausteine?

Wir lösen die Beispiele noch einmal mit `gleiche_auslassen=False`, also ohne die Abkürzung, und zählen die Aufrufe der Suche.
Ohne die Abkürzung werden gleiche Bausteine in jeder Reihenfolge ausprobiert. Bei Beispiel 3 (kein Ergebnis, alles muss
durchprobiert werden) sieht man das gut. Beispiel 5 lassen wir weg, ohne die Abkürzung dauert es viel zu lange.

In [ ]:
vergleich = {}
for nummer in [1, 2, 3]:
    k, b, g, mit = ergebnisse[nummer]
    g_ohne = loese(k, b, gleiche_auslassen=False)
    ohne = zaehler['aufrufe']
    vergleich[nummer] = (mit, ohne)
    print('raetsel' + str(nummer) + ': mit Abkürzung', mit, 'Aufrufe, ohne', ohne, 'Aufrufe')

raetsel1: mit Abkürzung 143 Aufrufe, ohne 1919 Aufrufe
raetsel2: mit Abkürzung 19 Aufrufe, ohne 44 Aufrufe
raetsel3: mit Abkürzung 393 Aufrufe, ohne 1422 Aufrufe


Die Lösungshinweise nennen weitere Verbesserungen, z. B. beim ersten Baustein nur eine Drehung zu probieren
(die anderen entstehen durch Drehen der ganzen Kiste), was bei würfelförmigen Kisten gilt.

## Zum Schluss: Dasselbe mit MiniZinc

Statt selbst zu suchen, kann man die Regeln **beschreiben** und einen Solver suchen lassen. Hier ist die Idee:
Jeder Baustein bekommt als Unbekannte seine **Drehung** (1 bis 6) und seine **Ecke** `(x, y, z)`. Die Bedingungen:

- Jeder Baustein liegt ganz in der Kiste.
- Das Gold (ein Baustein `1 × 1 × 1` mit fester Ecke in der Mitte) gehört dazu.
- Keine zwei Bausteine überlappen sich. Dafür gibt es die fertige Bedingung `diffn_k(p, d)`: `p` sind die Ecken, `d` die Maße
  der Quader. Weil das Volumen der Bausteine gleich dem der Kiste ist (das prüfen wir in Python vorher), bleibt dann keine Lücke.

Die Modelle sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den [MiniZinc Playground](https://play.minizinc.org/)).

Das Modell (`kiste.mzn`):

```minizinc
include "globals.mzn";
int: kx; int: ky; int: kz;                      % Maße der Kiste
int: n;                                         % Anzahl der Bausteine (ohne Gold)
array[1..n, 1..3] of int: mass;                 % Maße der Bausteine
set of int: BLOECKE = 1..n + 1;                 % Baustein n + 1 ist das Gold

% alle 6 Reihenfolgen der drei Maße (jede Zeile eine Drehung)
array[1..6, 1..3] of int: dreh = array2d(1..6, 1..3, [1,2,3, 1,3,2, 2,1,3, 2,3,1, 3,1,2, 3,2,1]);

array[1..n] of var 1..6: o;                     % welche Drehung
array[BLOECKE, 1..3] of var 1..max([kx, ky, kz]): d;       % Maße nach der Drehung
array[BLOECKE, 1..3] of var 0..max([kx, ky, kz]) - 1: p;   % Ecke (kleinste Koordinaten)

constraint forall(i in 1..n, j in 1..3)(d[i, j] = mass[i, dreh[o[i], j]]);
% Gold: 1 x 1 x 1 genau in der Mitte
constraint forall(j in 1..3)(d[n + 1, j] = 1 /\ p[n + 1, j] = [kx, ky, kz][j] div 2);
% alle Bausteine liegen in der Kiste
constraint forall(i in BLOECKE)(p[i, 1] + d[i, 1] <= kx /\ p[i, 2] + d[i, 2] <= ky /\ p[i, 3] + d[i, 3] <= kz);
% keine zwei Bausteine überlappen (Volumen ist gleich, also bleibt keine Lücke)
constraint diffn_k(p, d);

solve satisfy;
output ["Baustein \(i): Ecke \(p[i,1]) \(p[i,2]) \(p[i,3]), Maße \(d[i,1]) \(d[i,2]) \(d[i,3])\n" | i in BLOECKE];
```

Die Datendatei erzeugt diese Python-Zelle aus Kiste und Bausteinen:

In [ ]:
def als_dzn(kiste, bausteine):
    zahlen = []
    for baustein in bausteine:
        zahlen = zahlen + [str(m) for m in baustein]
    return ('kx = ' + str(kiste[0]) + ';\nky = ' + str(kiste[1]) + ';\nkz = ' + str(kiste[2]) + ';\nn = ' + str(len(bausteine)) +
            ';\nmass = array2d(1..' + str(len(bausteine)) + ', 1..3, [' + ', '.join(zahlen) + ']);')


k, b, g, _ = ergebnisse[2]
print(als_dzn(k, b))

kx = 3;
ky = 3;
kz = 3;
n = 6;
mass = array2d(1..6, 1..3, [3, 3, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 3]);


Beispiel 2 (`raetsel2.txt`) mit Chuffed:

**Ausgabe:**

```
Baustein 1: Ecke 2 0 0, Maße 1 3 3
Baustein 2: Ecke 0 0 0, Maße 1 3 3
Baustein 3: Ecke 1 2 1, Maße 1 1 1
Baustein 4: Ecke 1 0 1, Maße 1 1 1
Baustein 5: Ecke 1 0 0, Maße 1 3 1
Baustein 6: Ecke 1 0 2, Maße 1 3 1
Baustein 7: Ecke 1 1 1, Maße 1 1 1
----------
```

Baustein 7 ist das Gold in der Mitte (Ecke `1 1 1`). Beispiel 3 hat keine Lösung, das meldet der Solver auch:

**Ausgabe:**

```
=====UNSATISFIABLE=====
```

**Grenzen:** Die 3 × 3 × 3-Kisten schafft MiniZinc in Sekunden. Bei den 5 × 5 × 5-Kisten brauchte Gecode für Beispiel 4
fast eine Minute, Chuffed war nach mehr als 100 Sekunden nicht fertig. Die gezielte Suche aus diesem Notebook ist hier viel
schneller, weil sie ausnutzt, dass in der ersten freien Zelle die Ecke eines Bausteins liegen muss.